In [1]:
import os
import gurobipy as gp
from gurobipy import GRB

os.environ['GRB_LICENSE_FILE'] = r'C:\Users\anabe\gurobi.lic'


In [2]:
# 1. DADOS

id_instancia = 16

# conflitos
conflitos = [(0, 12), (0, 16), (0, 5), (0, 1), (0, 10), (0, 4), (0, 18), (0, 7), (0, 9), (0, 20), (0, 2), (1, 14), (1, 13), (1, 4), (1, 17), (1, 7), (1, 2), (1, 10), (1, 12), (1, 3), (1, 8), (1, 5), (1, 6), (2, 9), (2, 4), (2, 3), (2, 18), (2, 8), (2, 11), (2, 25), (2, 16), (2, 5), (3, 17), (3, 5), (3, 14), (3, 7), (3, 12), (3, 9), (4, 5), (4, 6), (4, 16), (4, 20), (4, 22), (4, 7), (4, 17), (5, 9), (5, 6), (5, 10), (5, 15), (6, 11), (6, 8), (6, 7), (6, 19), (6, 24), (7, 12), (7, 16), (7, 13), (8, 10), (8, 27), (8, 15), (8, 13), (9, 11), (9, 17), (9, 19), (9, 26), (10, 20), (10, 13), (11, 15), (11, 19), (13, 23), (14, 18)]


# frente e atras
frente = [7, 31, 23, 27, 8, 13]
atras = [19, 29, 6]

#proximidade
J = [(1, 26), (3, 19), (7, 14)]  

# capacidade das fileiras
#apenas alterar aqui para deixar carteiras vazias!! carteiras > alunos
fileiras_cap = [6, 7, 4, 5, 6, 7]


# infos
num_alunos = 32
alunos = list(range(num_alunos))
num_fileiras = len(fileiras_cap)

# parametros
P = 2*((len(conflitos) * max(fileiras_cap)) + 1)  
d_min = 2                

In [3]:
# 2. VARIAVEIS

model = gp.Model(f"Instancia {id_instancia}")

# x[i, L, k] é 1 se aluno i está na fileira L posição k
x = {}
for i in alunos:
    for L in range(num_fileiras):
        for k in range(fileiras_cap[L]):
            x[i, L, k] = model.addVar(vtype=GRB.BINARY, name=f"x_{i}_{L}_{k}")

# y[i, j] é 1 se a aresta de conflito entre i e j está ativa
y = {}
for (i, j) in conflitos:
    y[i, j] = model.addVar(vtype=GRB.BINARY, name=f"y_{i}_{j}")

# w[L, i, j, k, z] lineariza o produto x[i,L,k] * x[j,L+1,z]
w = {}
for L in range(num_fileiras - 1):
    for (i, j) in conflitos:
        for k in range(fileiras_cap[L]):
            for z in range(fileiras_cap[L+1]):
                w[L, i, j, k, z] = model.addVar(vtype=GRB.BINARY, name=f"w_{L}_{i}_{j}_{k}_{z}")

Set parameter Username
Set parameter LicenseID to value 2709062
Academic license - for non-commercial use only - expires 2026-09-15


In [4]:
# 3. F.O.

# max dist horizontal e min numero de conflitos
obj = gp.quicksum(
    (abs(z - k) * w[L, i, j, k, z]) - (P * y[i, j])
    for L in range(num_fileiras - 1)
    for (i, j) in conflitos
    for k in range(fileiras_cap[L])
    for z in range(fileiras_cap[L+1])
)
model.setObjective(obj, GRB.MAXIMIZE)

In [5]:
# 4. S.T.

# cada aluno deve ocupar um assento
for i in alunos:
    model.addConstr(gp.quicksum(x[i, L, k] for L in range(num_fileiras) for k in range(fileiras_cap[L])) == 1)

# cada assento deve ter no máximo um aluno
for L in range(num_fileiras):
    for k in range(fileiras_cap[L]):
        model.addConstr(gp.quicksum(x[i, L, k] for i in alunos) <= 1)

# se i e j em conflito estão na mesma fileira a dist >= d_min
for (i, j) in conflitos:
    for L in range(num_fileiras):
        for k in range(fileiras_cap[L]):
            for z in range(k + 1, fileiras_cap[L]):
                model.addConstr(z - k >= (x[i, L, k] + x[j, L, z] - 1) * d_min)

# definição de aresta ativa (y)
for (i, j) in conflitos:
    for L in range(num_fileiras - 1):
        for k in range(fileiras_cap[L]):
            for z in range(fileiras_cap[L+1]):
                # i em L,k e j em L+1,z -> aresta fica ativa
                model.addConstr(y[i, j] >= x[i, L, k] + x[j, L+1, z] - 1)
                model.addConstr(y[i, j] >= x[j, L, k] + x[i, L+1, z] - 1)

                # def W
                model.addConstr(w[L, i, j, k, z] <= x[i, L, k])
                model.addConstr(w[L, i, j, k, z] <= x[j, L+1, z])
                model.addConstr(w[L, i, j, k, z] >= x[i, L, k] + x[j, L+1, z] - 1)

# alunos na lista frente: k=1 ou k=2
for i_frente in frente:
    model.addConstr(gp.quicksum(x[i_frente, L, k] for L in range(num_fileiras) for k in [0, 1]) == 1)

# alunos na lista atras: duas últimas posições
for i_atras in atras:
    model.addConstr(gp.quicksum(x[i_atras, L, k] for L in range(num_fileiras) for k in [fileiras_cap[L]-2, fileiras_cap[L]-1]) == 1)
    
# NOVA RESTRIÇÃO: proximidade obrigatória (S)
for (i, j) in J:
    for L in range(num_fileiras):
        for k in range(fileiras_cap[L]):
            vizinhos = []
            
            # Lado
            if k > 0: 
                vizinhos.append(x[j, L, k-1]) # Esquerda
            if k < fileiras_cap[L] - 1: 
                vizinhos.append(x[j, L, k+1]) # Direita
                
            # Frente e Trás
            if L > 0 and k < fileiras_cap[L-1]: 
                vizinhos.append(x[j, L-1, k]) # Frente
            if L < num_fileiras - 1 and k < fileiras_cap[L+1]: 
                vizinhos.append(x[j, L+1, k]) # Atrás
            
            # Se o aluno i sentar em (L, k), o aluno j DEVE estar em um dos vizinhos
            model.addConstr(x[i, L, k] <= gp.quicksum(vizinhos), name=f"dupla_{i}_{j}_{L}_{k}")

In [ ]:
# 5. RESULTADO

model.Params.TimeLimit = 300
model.optimize()

# --- F. Verificação de Status, Tempo e Contagem de Conflitos ---
    
status_map = {
        GRB.OPTIMAL: "SOLUÇÃO ÓTIMA",
        GRB.TIME_LIMIT: "LIMITE DE TEMPO ATINGIDO",
        GRB.INFEASIBLE: "MODELO INVIÁVEL",
        GRB.INF_OR_UNBD: "INVIÁVEL OU ILIMITADO"
    }
status_msg = status_map.get(model.Status, f"DESCONHECIDO")

    # Aqui imprimimos o NÚMERO (model.Status) e depois o texto
print(f"\nINSTÂNCIA {id_instancia:02d} | Status: {model.Status} ({status_msg})")
print(f"Tempo de Execução: {model.Runtime:.2f} s")

if model.SolCount > 0:
        print(f"Valor da FO: {model.ObjVal:.2f}")
        print(f"Melhor Limite Teórico (Best Bound): {model.ObjBound:.2f}")
        print(f"MIPGap: {model.MIPGap * 100:.4f}%")
        
        arestas_ativas = sum(1 for (i, j) in conflitos if y[i, j].X > 0.5)
        
        if arestas_ativas > 0:
            print(f"RESULTADO: {arestas_ativas} aresta(s) de conflito permanecem ativa(s).")
        else:
            print("RESULTADO: Zero conflitos entre fileiras adjacentes.")
else:
        print("Valor da FO: N/A (Inviável)")
        print("RESULTADO: Sem solução.")
    
print("-" * 50)

Set parameter TimeLimit to value 300
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-1035G1 CPU @ 1.00GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 63939 rows, 12693 columns and 164799 nonzeros
Model fingerprint: 0x8ba34c6e
Variable types: 0 continuous, 12693 integer (12693 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [1e+00, 2e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 8e+00]
Presolved: 30028 rows, 7737 columns, 79416 nonzeros

Continuing optimization...


Explored 1 nodes (87114 simplex iterations) in 0.02 seconds (0.00 work units)
Thread count was 8 (of 8 available processors)

Solution count 7: -0 -161347 -161348 ... -2.74297e+06
No other solutions better than -0

Optimal solution found (tolerance 1.00e-04)
Best object